# Linear Regression Health Costs Calculator

An 80/20 split keeps the final test set unseen. Categorical encoding and smoking interactions feed a small neural regressor; early stopping uses only a validation subset of the training rows.

Prepared with AI assistance for an agentic AI class assignment. Starter notebook and original challenge tests: [freeCodeCamp](https://github.com/freeCodeCamp/boilerplate-linear-regression-health-costs-calculator). The supplied final test cell is preserved. Run all cells from top to bottom.

In [1]:
import os
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
os.environ.setdefault('TF_NUM_INTRAOP_THREADS', '2')
os.environ.setdefault('TF_NUM_INTEROP_THREADS', '2')
DATA_DIR = Path(os.environ.get('FCC_DATA_DIR', 'data')).resolve()
ARTIFACT_DIR = Path(os.environ.get('FCC_ARTIFACT_DIR', 'artifacts')).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def data_file(name, relative_url, extract=False):
    # The official freeCodeCamp CDN repository is a public mirror of the datasets.
    path = DATA_DIR / name
    if not path.exists() or not path.stat().st_size:
        url = 'https://raw.githubusercontent.com/freeCodeCamp/cdn/main/build/project-data/' + relative_url
        with urlopen(url, timeout=120) as response:
            contents = response.read()
        temporary = path.with_suffix(path.suffix + '.part')
        temporary.write_bytes(contents)
        temporary.replace(path)
    if extract:
        with ZipFile(path) as archive:
            for member in archive.namelist():
                (DATA_DIR / member).resolve().relative_to(DATA_DIR)
            archive.extractall(DATA_DIR)
    return path

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(42)
try:
    tf.config.threading.set_intra_op_parallelism_threads(2)
    tf.config.threading.set_inter_op_parallelism_threads(2)
except RuntimeError:
    pass  # The runtime may already be initialized when a notebook cell is rerun.

def batches(x, y=None, training=False, batch_size=64):
    contents = x if y is None else (x, y)
    ds = tf.data.Dataset.from_tensor_slices(contents)
    if training:
        ds = ds.shuffle(len(x), seed=42)
    options = tf.data.Options()
    options.threading.private_threadpool_size = 2
    options.threading.max_intra_op_parallelism = 2
    return ds.batch(batch_size).with_options(options).prefetch(1)

In [3]:
dataset = pd.read_csv(data_file('insurance.csv', 'health-costs/insurance.csv'))
print(dataset.tail())

      age     sex   bmi  children smoker     region  expenses
1333   50    male  31.0         3     no  northwest  10600.55
1334   18  female  31.9         0     no  northeast   2205.98
1335   18  female  36.9         0     no  southeast   1629.83
1336   21  female  25.8         0     no  southwest   2007.95
1337   61  female  29.1         0    yes  northwest  29141.36


In [4]:
encoded = dataset.copy()
encoded['sex'] = encoded['sex'].map({'female':0, 'male':1})
encoded['smoker'] = encoded['smoker'].map({'no':0, 'yes':1})
encoded['region'] = pd.Categorical(encoded['region'],
    categories=['northeast','northwest','southeast','southwest'])
encoded = pd.get_dummies(encoded, columns=['region'], dtype='float32')
encoded['smoker_bmi'] = encoded['smoker'] * encoded['bmi']
encoded['smoker_age'] = encoded['smoker'] * encoded['age']
encoded['smoker_obese'] = encoded['smoker'] * (encoded['bmi'] >= 30).astype(float)
encoded = encoded.astype('float32')

train_dataset = encoded.sample(frac=0.8, random_state=42)
test_dataset = encoded.drop(train_dataset.index)
train_labels = train_dataset.pop('expenses')
test_labels = test_dataset.pop('expenses')
validation_rows = train_dataset.sample(frac=0.2, random_state=42).index
fit_rows = train_dataset.index.difference(validation_rows)
x_train = train_dataset.loc[fit_rows].to_numpy()
x_val = train_dataset.loc[validation_rows].to_numpy()
normalizer = layers.Normalization()
normalizer.adapt(x_train)
core = keras.Sequential([
    keras.Input(shape=(train_dataset.shape[1],)), normalizer,
    layers.Dense(64, activation='relu'), layers.Dense(32, activation='relu'),
    layers.Dense(1)
])
core.compile(optimizer=keras.optimizers.Adam(0.003), loss='mae')
history = core.fit(
    batches(x_train, train_labels.loc[fit_rows].to_numpy()/10000, training=True, batch_size=32),
    validation_data=batches(x_val, train_labels.loc[validation_rows].to_numpy()/10000, batch_size=32),
    epochs=250, verbose=2,
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=30,
                                            restore_best_weights=True)]
)
model = keras.Sequential([keras.Input(shape=(train_dataset.shape[1],)), core,
                          layers.Rescaling(10000.0)])
model.compile(optimizer='adam', loss='mse', metrics=['mae','mse'])
print(f'Training rows: {len(train_dataset)}; held-out test rows: {len(test_dataset)}')

/workspace/scratch/19945c1fc1e3/ml_runtime/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
Epoch 1/250
27/27 - 1s - 28ms/step - loss: 0.6926 - val_loss: 0.2249
Epoch 2/250
27/27 - 0s - 1ms/step - loss: 0.2313 - val_loss: 0.1485
Epoch 3/250
27/27 - 0s - 1ms/step - loss: 0.1891 - val_loss: 0.1203
Epoch 4/250
27/27 - 0s - 1ms/step - loss: 0.1819 - val_loss: 0.1148
Epoch 5/250
27/27 - 0s - 1ms/step - loss: 0.1750 - val_loss: 0.1085
Epoch 6/250
27/27 - 0s - 1ms/step - loss: 0.1711 - val_loss: 0.1121
Epoch 7/250
27/27 - 0s - 1ms/step - loss: 0.1660 - val_loss: 0.1062
Epoch 8/250
27/27 - 0s - 1ms/step - loss: 0.1744 - val_loss: 0.1190
Epoch 9/250
27/27 - 0s - 1ms/step - loss: 0.1773 - val_loss: 0.1010
Epoch 10/250
27/27 - 0s - 94

In [5]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)


9/9 - 0s - 18ms/step - loss: 24671218.0000 - mae: 1637.8068 - mse: 24671218.0000
Testing set Mean Abs Error: 1637.81 expenses
You passed the challenge. Great job!
1/9 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 


In [6]:
assert mae < 3500, f'MAE {mae:.2f} does not meet the official requirement.'
model.save(ARTIFACT_DIR / 'health_costs.keras')
print(f'PASS: held-out MAE = ${mae:.2f}')

PASS: held-out MAE = $1637.81
